# Apartment Price Prediction

This notebook predicts apartment price from floor area using linear regression.
Prices are in thousands of PLN. The supplied teaching dataset has 10 apartments.
All five independent exercises are completed below.

Source: [PJATK tutorial](https://github.com/PRO-D-PJATK/tutorials/tree/main/google-colab-linear-regression).
The dataset is a small teaching sample, not a current housing market dataset.

## Run in Google Colab
Upload this notebook to Colab, then choose **Runtime → Run all**.
The data cell reads the included CSV locally, or the original tutorial CSV if
running in a new Colab session. No account credentials are needed in the code.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import LeaveOneOut, cross_val_predict

print("Python:", sys.version.split()[0])


Python: 3.12.14


## 1. Load and inspect the data

In [2]:
data_file = Path("apartments.csv")
source = str(data_file) if data_file.exists() else "https://raw.githubusercontent.com/PRO-D-PJATK/tutorials/main/google-colab-linear-regression/apartments.csv"
df = pd.read_csv(source)
assert list(df.columns) == ["area_m2", "price_thousands_pln"]
assert not df.isna().any().any()
assert (df > 0).all().all()
print("Rows:", len(df))
df


Rows: 10


,area_m2,price_thousands_pln
0,35,310
1,42,355
2,50,395
3,57,430
4,65,470
5,72,515
6,80,560
7,90,615
8,100,670
9,120,770


## 2. Train a linear model
The model has the form `price = a × area + b`.

In [3]:
X = df[["area_m2"]]
y = df["price_thousands_pln"]
model = LinearRegression()
model.fit(X, y)
print(f"Slope: {model.coef_[0]:.6f} thousand PLN per m²")
print(f"Intercept: {model.intercept_:.6f} thousand PLN")


Slope: 5.435188 thousand PLN per m²
Intercept: 122.558161 thousand PLN


## 3. Predict new apartment prices

In [4]:
new_apartments = pd.DataFrame({"area_m2": [40, 60, 80, 100]})
new_apartments["predicted_thousands_pln"] = model.predict(new_apartments)
new_apartments.round(2)


,area_m2,predicted_thousands_pln
0,40,339.97
1,60,448.67
2,80,557.37
3,100,666.08


## 4. Plot and save the regression line

In [5]:
area_grid = pd.DataFrame({"area_m2": np.linspace(X["area_m2"].min(), X["area_m2"].max(), 200)})
plt.figure(figsize=(7, 4))
plt.scatter(df["area_m2"], y, label="Apartments")
plt.plot(area_grid["area_m2"], model.predict(area_grid), color="orange", label="Linear regression")
plt.xlabel("Floor area (m²)")
plt.ylabel("Price (thousands of PLN)")
plt.title("Apartment area and price")
plt.legend()
plt.tight_layout()
plt.savefig("regression.png", dpi=160)
plt.show()


![Generated plot](regression.png)

## 5. Price per square metre
Multiply by 1,000 first because the source price is in thousands of PLN.

In [6]:
df["price_per_m2"] = df["price_thousands_pln"] * 1000 / df["area_m2"]
print(df.round(2).to_string(index=False))
plt.figure(figsize=(7, 4))
plt.plot(df["area_m2"], df["price_per_m2"], "o-")
plt.xlabel("Floor area (m²)")
plt.ylabel("Price per m² (PLN)")
plt.title("Unit price and apartment area")
plt.tight_layout()
plt.savefig("price_per_m2.png", dpi=160)
plt.show()


 area_m2  price_thousands_pln  price_per_m2
      35                  310       8857.14
      42                  355       8452.38
      50                  395       7900.00
      57                  430       7543.86
      65                  470       7230.77
      72                  515       7152.78
      80                  560       7000.00
      90                  615       6833.33
     100                  670       6700.00
     120                  770       6416.67


![Generated plot](price_per_m2.png)

In this sample, the price per square metre decreases as the area increases.
The total price still increases. These are different quantities.

## 6. Errors and a polynomial model
Add area squared as a second feature. Compare training errors and
leave-one-out cross-validation errors. With only 10 rows, leave-one-out lets
each row be tested once using a model fitted on the other nine rows.
Cross-validation refits every model; the held-out row is not used for fitting.


In [7]:
polynomial = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression())
polynomial.fit(X, y)
rows = []
for name, estimator in [("Linear", model), ("Polynomial degree 2", polynomial)]:
    fitted = estimator.predict(X)
    held_out = cross_val_predict(estimator, X, y, cv=LeaveOneOut())
    rows.append({
        "model": name,
        "training_MAE": mean_absolute_error(y, fitted),
        "training_MSE": mean_squared_error(y, fitted),
        "LOOCV_MAE": mean_absolute_error(y, held_out),
        "LOOCV_MSE": mean_squared_error(y, held_out),
        "LOOCV_RMSE": np.sqrt(mean_squared_error(y, held_out))
    })
metrics = pd.DataFrame(rows)
metrics.to_csv("model_comparison.csv", index=False)
print("MAE and RMSE: thousands of PLN; MSE: squared thousands of PLN")
print(metrics.round(4).to_string(index=False))
print("Lower cross-validation MAE:", metrics.loc[metrics["LOOCV_MAE"].idxmin(), "model"])


MAE and RMSE: thousands of PLN; MSE: squared thousands of PLN
              model  training_MAE  training_MSE  LOOCV_MAE  LOOCV_MSE  LOOCV_RMSE
             Linear        3.1559       12.2442     4.1616    22.7936      4.7743
Polynomial degree 2        2.6857       11.0285     4.7622    42.3895      6.5107
Lower cross-validation MAE: Linear


In [8]:
plt.figure(figsize=(7, 4))
plt.scatter(df["area_m2"], y, label="Apartments")
plt.plot(area_grid["area_m2"], model.predict(area_grid), label="Linear")
plt.plot(area_grid["area_m2"], polynomial.predict(area_grid), label="Polynomial degree 2")
plt.xlabel("Floor area (m²)")
plt.ylabel("Price (thousands of PLN)")
plt.title("Linear and polynomial fit")
plt.legend()
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=160)
plt.show()


![Generated plot](model_comparison.png)

## Conclusion
Area is strongly related to total price in this teaching sample. The slope
shows the estimated change in price for one extra square metre. The intercept
is part of the fitted equation; it is not a realistic valuation of a 0 m² home.

A polynomial can reduce training error simply by being more flexible. The
cross-validation table checks whether it also predicts held-out rows better.
Ten examples are too few to judge real market performance. Location, condition
and other important features are missing, and extrapolation beyond the observed
35–120 m² range is unreliable.
